In [ ]:
from grecco_sim.graph.utils.format import Format
import pandas as pd
from pathlib import Path

def load_and_process_data(p1, p2, year, output_csv_path):
    # --- 1. Daten einlesen ---
    solar = pd.read_csv(p1, sep=';', skipinitialspace=True)
    temp = pd.read_csv(p2, sep=';', skipinitialspace=True)

    # --- 2. Zeitformat umwandeln ---
    solar['Date'] = pd.to_datetime(solar['MESS_DATUM'], format='%Y%m%d%H%M')
    temp['Date'] = pd.to_datetime(temp['MESS_DATUM'], format='%Y%m%d%H%M')

    # --- 3. Auf das gewünschte Jahr filtern ---
    solar = solar[solar['Date'].dt.year == year]
    temp = temp[temp['Date'].dt.year == year]

    # --- 4. Nur relevante Spalten behalten ---
    solar = solar[['Date', 'GS_10']].copy()
    temp = temp[['Date', 'TT_10']].copy()

    # --- 5. Auf 15-Minuten-Raster bringen (per "nearest" join) ---
    target_times = pd.date_range(start=f'{year}-01-01', end=f'{year}-12-31 23:59', freq='15min')

    solar_15min = pd.merge_asof(
        pd.DataFrame({'Date': target_times}),
        solar.sort_values('Date'),
        on='Date',
        direction='nearest',
        tolerance=pd.Timedelta('7min')  # toleriert ±7 Minuten
    )

    temp_15min = pd.merge_asof(
        pd.DataFrame({'Date': target_times}),
        temp.sort_values('Date'),
        on='Date',
        direction='nearest',
        tolerance=pd.Timedelta('7min')
    )

    # --- 6. Zusammenführen und Export ---
    result = pd.DataFrame({
        'Date': target_times,
        'Solar Irradiance': solar_15min['GS_10'],
        'Outside Temperature': temp_15min['TT_10']
    })

    # Optional: Fehlende Werte rausfiltern (falls -999 oder NaN)
    result = result.replace(-999, pd.NA).dropna()

    # Speichern
    result.to_csv(output_csv_path, index=False)
    print(f"✅ Fertig! Datei gespeichert unter: {output_csv_path}")

# Beispielaufruf:
# load_and_process_data("p1.csv", "p2.csv", 2016, "output_2016.csv")

data_dir = Format().data_root / "weather"
p1 = data_dir / "dwd" / "sd_2020_2024_10.txt"
p2 = data_dir / "dwd" / "tu_2020_2024_10.txt"
year = 2023
output = data_dir / f"{year}_dwd.csv"

load_and_process_data(p1, p2, year, output)
